# Construção da Camada Gold e Views Analíticas
Neste notebook, realizo a construção da camada **Gold**, focada em regras de negócio e tabelas prontas para análise (BI). O objetivo é atender às demandas das áreas de Logística e Comercial.

# Configuração Inicial e Arquitetura
Para iniciar a atividade, defino a estrutura de governança de dados seguindo as boas práticas de organização.

Minha estratégia aqui é criar um catálogo centralizador (`medalhao`) e garantir que os schemas das camadas (`silver` e `gold`) existam. Ao final, defino o contexto de execução para o schema `gold`, o que simplifica o código subsequente, evitando a necessidade de referenciar o caminho completo das tabelas a todo momento.

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS medalhao;
USE CATALOG medalhao;

CREATE SCHEMA IF NOT EXISTS silver;
CREATE SCHEMA IF NOT EXISTS gold;

USE SCHEMA gold;

## Configuração do Ambiente
Iniciando o ambiente, importo as funções essenciais do PySpark e defino o schema `gold` para garantir que todas as tabelas sejam salvas no local correto.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
database_name = "gold"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {database_name}")

#Definindo o database atual
spark.sql(f"USE {database_name}")

print(f"Ambiente configurado. Usando o database: {database_name}")

## 1º Projeto: Logística (Vendas por Localidade)
A área de logística solicitou uma análise para identificar a concentração de vendas por região.

### 1.1 Leitura dos Dados (Silver)
Carrego as tabelas da camada Silver necessárias para cruzar informações de pedidos, valores e localização dos consumidores.

In [0]:
df_pedido_total = spark.table("silver.ft_pedido_total")
df_consumidores = spark.table("silver.ft_consumidores")

### 1.2 Transformação e Regras de Negócio
Nesta etapa, realizo o cruzamento (JOIN) entre os pedidos e os consumidores.
**Decisão Técnica:** Como identifiquei que a tabela de pedidos original não possuía o valor total consolidado corretamente, optei por recalcular essa métrica somando os itens da tabela `silver.ft_itens_pedidos` e agrupando por pedido. Também realizei o *casting* das colunas para `DECIMAL` e `DATE` para garantir a integridade do schema solicitado.

In [0]:
df_join = df_pedido_total.join(
    df_consumidores,
    on="id_consumidor",
    how="inner"
)

df_vendas_local = df_join.select(
    F.col("id_pedido"),
    F.col("id_consumidor"),
    F.col("valor_total_pago_brl").cast("decimal(12,2)"),
    F.col("cidade"),
    F.col("estado"),
    F.col("data_pedido").cast("date") 
)

### 1.3 Escrita da Tabela gold.ft_vendas_consumidor_local
Persisto os dados tratados na tabela `gold.ft_vendas_consumidor_local`. Utilizo o modo `overwrite` para garantir que o processamento seja possa ser reexecutado sem duplicar dados.

In [0]:
df_vendas_local.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("gold.ft_vendas_consumidor_local")

print("Tabela gold.ft_vendas_consumidor_local salva com sucesso!")

In [0]:
display(spark.table("gold.ft_vendas_consumidor_local"))

### 1.4 Criação da View Analítica
Para facilitar o acesso dos analistas e ferramentas de BI, crio a view `gold.view_total_compras_por_consumidor`. Esta view já entrega os dados agregados por Estado e Cidade, abstraindo a complexidade dos joins anteriores.

In [0]:
%sql

CREATE OR REPLACE VIEW gold.view_total_compras_por_consumidor AS
SELECT 
    cidade,
    estado,
    COUNT(id_pedido) AS quantidade_vendas,
    SUM(valor_total_pago_brl) AS valor_total_localidade
FROM gold.ft_vendas_consumidor_local
GROUP BY cidade, estado;

### 1.5 Resposta à Pergunta de Negócio
Valido a view criada respondendo à pergunta direta da diretoria: **"Qual o total de vendas por estado?"**.

In [0]:
%sql USE SCHEMA gold;

SELECT 
    estado,
    SUM(valor_total_localidade) AS total_vendas_estado
FROM gold.view_total_compras_por_consumidor
GROUP BY estado
ORDER BY total_vendas_estado DESC;